# Módulo 5 - Ingeniería de Variables

## Objetivo

En este notebook se construyen nuevas variables predictoras a partir del dataset integrado.

La ingeniería de variables busca enriquecer la información disponible para los modelos de aprendizaje automático mediante la creación de indicadores derivados que permitan representar mejor la realidad educativa de cada municipio.

Las nuevas variables fueron diseñadas teniendo en cuenta el contexto del problema de predicción de crisis educativa y la literatura sobre permanencia escolar, cobertura educativa y eficiencia interna.

Al finalizar este notebook se obtendrá un dataset enriquecido listo para el proceso de preprocesamiento y entrenamiento de modelos.

# 5.1 Carga del dataset integrado

En este notebook se utilizará el dataset integrado generado durante la etapa de preparación de los datos. Este conjunto contiene la información consolidada de los tres datasets originales (principal, ETC y PAE), obtenida mediante los procesos de limpieza, agregación e integración realizados en el notebook anterior.

Con el fin de preservar el dataset original, todas las transformaciones de ingeniería de variables se realizarán sobre una copia del mismo.

In [1]:
import pandas as pd
from pathlib import Path

# Ruta del proyecto
PROJECT_ROOT = Path.cwd().parent

# Cargar el dataset integrado
df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "dataset_integrado.csv"
)

# Verificar dimensiones
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

# Mostrar las primeras filas
df.head()

Filas: 15,707
Columnas: 77


,AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO,CÓDIGO_DEPARTAMENTO,DEPARTAMENTO,CÓDIGO_ETC,ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,...,REPROBACIÓN_TRANSICIÓN_ETC,REPROBACIÓN_PRIMARIA_ETC,REPROBACIÓN_SECUNDARIA_ETC,REPROBACIÓN_MEDIA_ETC,REPITENCIA_ETC,REPITENCIA_TRANSICIÓN_ETC,REPITENCIA_PRIMARIA_ETC,REPITENCIA_SECUNDARIA_ETC,REPITENCIA_MEDIA_ETC,BENEFICIARIOS_PAE
0,2024,5004,Abriaquí,5,Antioquia,3758.0,Antioquia (ETC),499.0,56.11,56.11,...,0.20,5.03,9.14,4.79,9.21,1.84,9.63,12.32,3.35,NaN
1,2024,15204,Cómbita,15,Boyacá,3769.0,Boyacá (ETC),1862.0,95.33,95.33,...,0.18,3.41,10.03,4.82,7.18,1.68,5.60,11.56,3.21,NaN
2,2024,99773,Cumaribo,99,Vichada,3832.0,Vichada (ETC),25239.0,50.70,50.70,...,0.00,16.52,9.77,4.01,17.06,0.91,22.17,12.59,4.08,NaN
3,2024,99624,Santa Rosalía,99,Vichada,3832.0,Vichada (ETC),1157.0,81.42,81.42,...,0.00,16.52,9.77,4.01,17.06,0.91,22.17,12.59,4.08,NaN
4,2024,99524,La Primavera,99,Vichada,3832.0,Vichada (ETC),2645.0,90.96,90.96,...,0.00,16.52,9.77,4.01,17.06,0.91,22.17,12.59,4.08,NaN


# 5.2 Creación de la variable *Brecha de Cobertura*

La cobertura bruta representa la relación entre el número total de estudiantes matriculados y la población en edad escolar, sin importar la edad de los estudiantes. Por su parte, la cobertura neta considera únicamente a los estudiantes que se encuentran en la edad esperada para cursar cada nivel educativo.

La diferencia entre ambas medidas constituye la **Brecha de Cobertura**, un indicador que permite aproximarse a fenómenos como la extraedad, la repitencia o el ingreso tardío al sistema educativo.

Matemáticamente:

$$
\text{Brecha de Cobertura} =
\text{Cobertura Bruta} -
\text{Cobertura Neta}
$$

Valores cercanos a cero indican que la mayoría de los estudiantes se encuentran en la edad esperada, mientras que valores altos pueden sugerir rezago escolar.

In [2]:
# Verificar que las columnas necesarias existan
columnas = ["COBERTURA_BRUTA", "COBERTURA_NETA"]

print(df[columnas].head())

   COBERTURA_BRUTA  COBERTURA_NETA
0            61.92           56.11
1           191.51           95.33
2            57.74           50.70
3            90.58           81.42
4            99.13           90.96


In [3]:
# Crear la variable Brecha de Cobertura
df["BRECHA_COBERTURA"] = (
    df["COBERTURA_BRUTA"] - df["COBERTURA_NETA"]
)

# Estadísticas descriptivas
df["BRECHA_COBERTURA"].describe()

count    15575.000000
mean        11.880832
std          9.106683
min        -69.580000
25%          7.690000
50%         10.700000
75%         14.670000
max         96.180000
Name: BRECHA_COBERTURA, dtype: float64

## Validación de la variable *Brecha de Cobertura*

Después de crear la nueva característica, se realiza una validación básica para conocer la cantidad de valores faltantes y la existencia de observaciones con brechas negativas. Esta revisión permite identificar posibles inconsistencias en los datos antes de continuar con la ingeniería de variables.

In [4]:
# Valores faltantes
print("Valores faltantes:", df["BRECHA_COBERTURA"].isna().sum())

# Brechas negativas
negativos = (df["BRECHA_COBERTURA"] < 0).sum()
print("Brechas negativas:", negativos)

# Porcentaje de brechas negativas
print(f"Porcentaje: {negativos / len(df) * 100:.2f}%")

Valores faltantes: 132
Brechas negativas: 341
Porcentaje: 2.17%


# 5.2 Ingeniería de variables

Con el objetivo de enriquecer la información disponible para el entrenamiento de modelos de aprendizaje automático, se construyeron nuevas variables derivadas a partir de los indicadores originales.

Estas variables buscan representar relaciones entre los diferentes indicadores educativos, el contexto territorial y la infraestructura del sistema educativo, permitiendo capturar patrones que no son evidentes en las variables originales.

Las variables creadas fueron:

- Brecha de cobertura.
- Brecha entre aprobación y reprobación.
- Índice de eficiencia interna.
- Presión sobre el sistema educativo.
- Índice de digitalización.
- Tamaño promedio de grupo normalizado.
- Peso relativo del municipio dentro de su ETC.
- Variable indicadora del periodo de pandemia.

In [5]:
# ============================================================
# INGENIERÍA DE VARIABLES
# ============================================================

# 1. Brecha de cobertura
df["BRECHA_COBERTURA"] = (
    df["COBERTURA_BRUTA"] - df["COBERTURA_NETA"]
)

# 2. Brecha aprobación - reprobación
df["BRECHA_APROBACION"] = (
    df["APROBACIÓN"] - df["REPROBACIÓN"]
)

# 3. Índice de eficiencia interna
df["INDICE_EFICIENCIA"] = (
    df["APROBACIÓN"] /
    (
        df["REPROBACIÓN"] +
        df["REPITENCIA"] +
        df["DESERCIÓN"]
    )
)

# 4. Presión sobre el sistema
df["PRESION_SISTEMA"] = (
    df["POBLACIÓN_5_16"] /
    df["TASA_MATRICULACIÓN_5_16"]
)

# 5. Índice de digitalización (dummy)
df["DIGITALIZADO"] = (
    df["SEDES_CONECTADAS_A_INTERNET"] >= 80
).astype("Int64")

# 6. Tamaño promedio de grupo normalizado
promedio_nacional = df["TAMAÑO_PROMEDIO_DE_GRUPO"].mean()

df["TAM_GRUPO_NORMALIZADO"] = (
    df["TAMAÑO_PROMEDIO_DE_GRUPO"] /
    promedio_nacional
)

# 7. Peso del municipio dentro de la ETC
df["PESO_MUNICIPIO_ETC"] = (
    df["POBLACIÓN_5_16"] /
    df["POBLACIÓN_5_16_ETC"]
)

# 8. Dummy pandemia
df["PANDEMIA"] = (
    df["AÑO"].isin([2020, 2021])
).astype("Int64")

print("Variables creadas correctamente.")

nuevas = [
    "BRECHA_COBERTURA",
    "BRECHA_APROBACION",
    "INDICE_EFICIENCIA",
    "PRESION_SISTEMA",
    "DIGITALIZADO",
    "TAM_GRUPO_NORMALIZADO",
    "PESO_MUNICIPIO_ETC",
    "PANDEMIA"
]

df[nuevas].describe(include="all")

Variables creadas correctamente.


,BRECHA_COBERTURA,BRECHA_APROBACION,INDICE_EFICIENCIA,PRESION_SISTEMA,DIGITALIZADO,TAM_GRUPO_NORMALIZADO,PESO_MUNICIPIO_ETC,PANDEMIA
count,15575.000000,15621.000000,1.539000e+04,1.559200e+04,15707.0,7572.000000,14573.000000,15707.0
mean,11.880832,86.771034,inf,inf,0.043929,1.000000,0.084924,0.142866
std,9.106683,8.714740,NaN,NaN,0.204945,0.321650,0.231148,0.349948
min,-69.580000,0.000000,7.944133e-01,6.774281e-03,0.0,0.125249,0.000641,0.0
25%,7.690000,81.120000,5.481320e+00,1.458009e+01,0.0,0.811188,0.007159,0.0
50%,10.700000,87.220000,8.258958e+00,3.230784e+01,0.0,1.000325,0.016538,0.0
75%,14.670000,94.050000,1.378852e+01,7.184328e+01,0.0,1.162325,0.037991,0.0
max,96.180000,100.000000,inf,inf,1.0,2.255742,1.000000,1.0


In [6]:
import numpy as np

# Reemplazar infinitos por NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Verificar
df[[
    "INDICE_EFICIENCIA",
    "PRESION_SISTEMA"
]].describe()

,INDICE_EFICIENCIA,PRESION_SISTEMA
count,15388.000000,1.558600e+04
mean,13.039313,2.084789e+03
std,22.645087,1.432391e+05
min,0.794413,6.774281e-03
25%,5.480932,1.457602e+01
50%,8.257870,3.229144e+01
75%,13.786201,7.171397e+01
max,1427.571429,1.037066e+07


In [7]:
from pathlib import Path

# Ruta de salida
output_path = PROJECT_ROOT / "data" / "processed" / "dataset_modelado.csv"

# Guardar
df.to_csv(output_path, index=False)

print("Dataset guardado correctamente.")
print(output_path)

Dataset guardado correctamente.
/home/juan/Downloads/educational_crisis_ai/educational_crisis_ai/data/processed/dataset_modelado.csv
